# Day 17 - Bayesian Optimization (Optuna + XGBoost)


## 학습 목표
- Bayesian Optimization 개념
- Optuna로 XGBoost 하이퍼파라미터 최적화


## 1. 데이터 준비


In [ ]:
import numpy as np
from sklearn.datasets import load_diabetes, load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, accuracy_score
import xgboost as xgb
import optuna
import warnings
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# 회귀
data_r = load_diabetes()
X_r = StandardScaler().fit_transform(data_r.data)
y_r = data_r.target
X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(X_r, y_r, test_size=0.2, random_state=42)

# 분류
data_c = load_breast_cancer()
X_c = StandardScaler().fit_transform(data_c.data)
y_c = data_c.target
X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(X_c, y_c, test_size=0.2, random_state=42, stratify=y_c)
print("Data ready")


## 2. Optuna - XGBoost 회귀 튜닝


In [ ]:
def objective_reg(trial):
    params = {
        'max_depth': trial.suggest_int('max_depth', 2, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
    }
    model = xgb.XGBRegressor(**params, random_state=42, n_jobs=-1, verbosity=0)
    scores = cross_val_score(model, X_tr_r, y_tr_r, cv=3, scoring='r2', n_jobs=-1)
    return scores.mean()

study_reg = optuna.create_study(direction='maximize')
study_reg.optimize(objective_reg, n_trials=30, show_progress_bar=False)
print("Best R2 (CV):", round(study_reg.best_value, 4))
print("Best params:", study_reg.best_params)

best_reg = xgb.XGBRegressor(**study_reg.best_params, random_state=42, n_jobs=-1, verbosity=0)
best_reg.fit(X_tr_r, y_tr_r)
print("Test R2:", round(r2_score(y_te_r, best_reg.predict(X_te_r)), 4))


## 3. Optuna - XGBoost 분류 튜닝


In [ ]:
def objective_clf(trial):
    params = {
        'max_depth': trial.suggest_int('max_depth', 2, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
    }
    model = xgb.XGBClassifier(**params, random_state=42, n_jobs=-1, verbosity=0,
                               use_label_encoder=False, eval_metric='logloss')
    scores = cross_val_score(model, X_tr_c, y_tr_c, cv=3, scoring='accuracy', n_jobs=-1)
    return scores.mean()

study_clf = optuna.create_study(direction='maximize')
study_clf.optimize(objective_clf, n_trials=30, show_progress_bar=False)
print("Best Acc (CV):", round(study_clf.best_value, 4))
print("Best params:", study_clf.best_params)

best_clf = xgb.XGBClassifier(**study_clf.best_params, random_state=42, n_jobs=-1, verbosity=0,
                              use_label_encoder=False, eval_metric='logloss')
best_clf.fit(X_tr_c, y_tr_c)
print("Test Acc:", round(accuracy_score(y_te_c, best_clf.predict(X_te_c)), 4))
